In [ ]:
! pip install langchain-text-splitters langchain_chroma langchain_community langchain_experimental langchain_huggingface

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 125.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 102.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/

In [ ]:
from datasets import load_dataset
import os
import pandas as pd

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

import os
import shutil
from langchain_chroma import Chroma
from sentence_transformers import SentenceTransformer
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

/tmp/ipykernel_894/3062291713.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_894/3062291713.py:14: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


In [ ]:
# ### Load the hugging face
# dataset = load_dataset("lucadiliello/newsqa")
# dataset_train = dataset['train'].select(range(0,3000))
# dataset_val = dataset['validation'].select(range(0,1000))
# dataset_test = dataset['validation'].select(range(1000,2000))

# ### Converting the hugging face dataset to dataframe
# dataset_train_df = dataset_train.to_pandas()[['context','question','answers']]
# dataset_val_df = dataset_val.to_pandas()[['context','question','answers']]
# dataset_test_df = dataset_test.to_pandas()[['context','question','answers']]

# base_data_path = '/content/drive/MyDrive/Colab Notebooks/RAG/Data'
# dataset_train_df.to_excel(os.path.join(base_data_path,'train.xlsx'))
# dataset_val_df.to_excel(os.path.join(base_data_path,'val.xlsx'))
# dataset_test_df.to_excel(os.path.join(base_data_path,'test.xlsx'))



In [ ]:
base_data_path = '/content/drive/MyDrive/Colab Notebooks/RAG/Data'
train_dataframe = pd.read_excel(os.path.join(base_data_path,'train.xlsx'))
val_dataframe = pd.read_excel(os.path.join(base_data_path,'val.xlsx'))
test_dataframe = pd.read_excel(os.path.join(base_data_path,'test.xlsx'))


print("Train Dataframe Shape:---",train_dataframe.shape)
print("Val Dataframe Shape:---",val_dataframe.shape)
print("test Dataframe Shape:---",test_dataframe.shape)


Train Dataframe Shape:--- (3000, 4)
Val Dataframe Shape:--- (1000, 4)
test Dataframe Shape:--- (1000, 4)


In [ ]:
train_dataframe.head()

,Unnamed: 0,context,question,answers
0,0,"NEW DELHI, India (CNN) -- A high court in nort...",What was the amount of children murdered?,['19']
1,1,"NEW DELHI, India (CNN) -- A high court in nort...",When was Pandher sentenced to death?,['February.']
2,2,"NEW DELHI, India (CNN) -- A high court in nort...",The court aquitted Moninder Singh Pandher of w...,['rape and murder']
3,3,"NEW DELHI, India (CNN) -- A high court in nort...",who was acquitted,['Moninder Singh Pandher']
4,4,"NEW DELHI, India (CNN) -- A high court in nort...",who was sentenced,['Moninder Singh Pandher']


In [ ]:
def create_total_context():

  total_data = []
  for ind in range(0,len(train_dataframe)):
    temp_list = [i for i in train_dataframe['context'][ind].split("\n") if i]
    total_data.extend(temp_list)
  for ind in range(0,len(val_dataframe)):
    temp_list = [i for i in val_dataframe['context'][ind].split("\n") if i]
    total_data.extend(temp_list)
  for ind in range(0,len(test_dataframe)):
    temp_list = [i for i in test_dataframe['context'][ind].split("\n") if i]
    total_data.extend(temp_list)

  total_text = '\n'.join(list(set(total_data)))
  return total_text

## Chunking Stategy
##
* it splits text hierarchically using a predefined list of separators. Instead of blindly slicing text at an exact character or token count, it attempts to keep semantic units like paragraphs, sentences, and words together for as long as possible

#### SemanticChunker
* SemanticChunker in LangChain is a text splitting technique that creates chunks based on semantic meaning rather than fixed token counts or character lengths.

#### Hierarchical
* Hierarchical chunking in LangChain involves breaking documents into nested, multi-layered segments (e.g., parent-child relationships)

* parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

3. Setup the vector store and the document store
vectorstore = Chroma(collection_name="split_parents", embedding_function=OpenAIEmbeddings())
store = InMemoryStore()

4. Initialize the ParentDocumentRetriever
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

5. Add documents to the retriever
retriever.add_documents(docs)


#### Contextual Chunking
* Contextual Chunking in LangChain solves standard RAG's "out-of-context" problem by prepending document-level summaries and surrounding section context to individual pieces of text.

In [ ]:
def chunking_RecursiveCharacterTextSplitter(chunk_size,chunk_overlap):
  total_text = create_total_context()
  text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap,is_separator_regex=False)
  texts = text_splitter.split_text(total_text)

  ### Creating langchain Document Object
  Documents = []
  for i in range(0,len(texts)):
    Documents.append(Document(page_content=texts[i], metadata={"chunk_ind": i+1}))
  return Documents

def chunking_semantic(embeddings , percentile = 80):

  total_text = create_total_context()
  splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=percentile
  )

  docs = splitter.create_documents([total_text])
  return docs



In [ ]:
def chunking_vector_creation(chunk_size,chunk_overlap,model_name):

  embeddings = HuggingFaceEmbeddings(model_name=model_name)
  path = '/content/drive/MyDrive/Colab Notebooks/RAG/vector_store/Chroma'



  ### Chunking Strategy
  # Documents  = chunking_RecursiveCharacterTextSplitter(chunk_size,chunk_overlap)
  Documents = chunking_semantic(embeddings , percentile = 80)

  if os.path.exists(path):
    print("Old vector store deleted Successfully")
    shutil.rmtree(path)


  vector_store = Chroma.from_documents(
      documents=Documents,
      embedding=embeddings,
      persist_directory=path,
      collection_name="ai_knowledge_base"
  )
  print("**** Vector Store Successfully Created*****")


In [ ]:
chunk_size = 1000
chunk_overlap = 100
model_name_basic = 'all-MiniLM-L6-v2'
model_qwen = 'Qwen/Qwen3-VL-Embedding-2B'

chunking_vector_creation(chunk_size,chunk_overlap,model_name_basic)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/770 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

In [ ]:
import os
import pandas as pd
base_data_path = '/content/drive/MyDrive/Colab Notebooks/RAG/Data'
train_dataframe = pd.read_excel(os.path.join(base_data_path,'train.xlsx'))
train_dataframe.head()

In [ ]:
train_dataframe['question'].tolist()[:10]